### Miguel Baños Baladrón
### Miguel Pérez Francos
### Rodrigo Touceda Tapias


# Proyecto de investigación: Reconocimiento de Actividades Humanas (HAR)con Sensores Inerciales

# Importaciones

In [1]:
include("setup.jl")
include("helpers.jl")
include("wrappers.jl")

# Usamos JLD2 para recuperar el estado exacto del preprocesamiento.
if isfile("datos_procesados.jld2")
    JLD2.@load "datos_procesados.jld2" df_trainval df_test X_trainval y_trainval folds_trainval
    println("Datos cargados correctamente.")
    println("   - Registros de entrenamiento: $(nrow(df_trainval))")
    println("   - Estructura de folds recuperada: $(typeof(folds_trainval))")
else
    error("No se encontró 'datos_procesados.jld2'. Ejecuta primero Preprocess.ipynb.")
end

Entorno cargado correctamente. Constantes: SEED=104, N_FEATURES=561
Entorno cargado correctamente. Constantes: SEED=104, N_FEATURES=561
Datos cargados correctamente.
   - Registros de entrenamiento: 9205
   - Estructura de folds recuperada: Vector{Tuple{Vector{Int64}, Vector{Int64}}}


# Función para ejecutar modelos

### Esta función tiene como objetivo sistematizar la evaluación de combinaciones entre técnias de selección de características, reducción de dimensionalidad y clasificadores:
### - Para que cada técnica aparezca al menos una vez, calculamos el número máximo de iteraciones (usando la lista de los modelos, la más larga) y se asignan filtros y reducciones de forma cíclica.
### - Es reproducible, ya que convierte los diccionarios en listas ordenadas alfabéticamente.
### - Añadimos una lógica de checkpoint, para que en caso de que la ejecución se detenga antes de terminar, la reanude desde el error, en vez de volver a empezar. Guardamos las métricas en un CSV.
### - Extraemos accuracy, f1_score, balanced_accuracy media y por cada fold.

In [2]:
function run_models(dic_filtros, dic_reducciones, dic_modelos, output_file; 
                    X=X_trainval, y=y_trainval, folds=folds_trainval)
    
    # Convierte diccionarios a listas ordenadas para ejecución determinista
    list_filtros     = sort(collect(dic_filtros), by=x->x[1])
    list_reducciones = sort(collect(dic_reducciones), by=x->x[1])
    list_modelos     = sort(collect(dic_modelos), by=x->x[1])

    # Calcula longitud de cada lista de opciones
    n_filtros = length(list_filtros)
    n_reducciones = length(list_reducciones)
    n_modelos = length(list_modelos)

    # Define el número máximo de iteraciones para cubrir todas las técnicas
    max_iter = max(n_filtros, n_reducciones, n_modelos)
    
    # Comprueba si existe archivo previo para reanudar o iniciar de cero
    if isfile(output_file)
        results_df = CSV.read(output_file, DataFrame)
        # Crea conjunto de identificadores para evitar repetir experimentos
        combinaciones_hechas = Set([
            (string(r.Filter), string(r.Reduction), string(r.Model)) 
            for r in eachrow(results_df)
        ])
    else
        # Inicializa DataFrame vacío con las columnas de métricas
        results_df = DataFrame(
            Filter = String[], Reduction = String[], Model = String[],
            Accuracy_Mean = Float64[], 
            F1_Score = Float64[], 
            B_Accuracy_mean = Float64[],
            B_accuracy_list = String[]
        )
        combinaciones_hechas = Set{Tuple{String, String, String}}()
    end
    
    # Define métricas: Accuracy, F1 y Balanced Accuracy
    measures = [accuracy, multiclass_f1score, balanced_accuracy]

    println("Iniciando ejecución: $max_iter experimentos totales.")

    # Bucle principal iterando cíclicamente
    for i in 1:max_iter
        # Selección modular de componentes para cubrir todas las opciones
        pair_filt = list_filtros[(i - 1) % n_filtros + 1]
        pair_red  = list_reducciones[(i - 1) % n_reducciones + 1]
        pair_mod  = list_modelos[(i - 1) % n_modelos + 1]

        # Desempaqueta nombres y objetos
        filt_name, filt_model = pair_filt
        red_name, red_model   = pair_red
        mod_name, mod_model   = pair_mod

        # Salta iteración si la combinación ya existe
        if (filt_name, red_name, mod_name) in combinaciones_hechas
            continue 
        end

        println("\nEvaluando ($i/$max_iter): [$filt_name] + [$red_name] + [$mod_name]")
        
        # Construye el pipeline con las técnicas seleccionadas
        pipe = PersonalizedPipeline(
            scaler    = MyMinMaxScaler(), 
            filter    = filt_model,      
            reduction = red_model,       
            clf       = mod_model        
        )
        
        try
            # Instancia la 'machine' de MLJ y ejecuta evaluación
            mach = machine(pipe, X, y) 
            evaluation = evaluate!(
                mach, 
                resampling = folds, 
                measures = measures, 
                verbosity = 0,
                acceleration = CPUThreads() 
            )
            
            # Extrae medias de las métricas
            acc_mean = evaluation.measurement[1]
            f1_mean   = evaluation.measurement[2]
            b_acc_mean   = evaluation.measurement[3]
            
            # Obtiene lista de Balanced Accuracy por fold y serializa a string
            b_acc_folds = evaluation.per_fold[3]
            b_acc_str = join(round.(b_acc_folds, digits=5), ";")
            
            println("Acc: $(round(acc_mean, digits=4)) | F1: $(round(f1_mean, digits=4)) | B_acc: $(round(b_acc_mean, digits=4))")
            
            # Guarda resultados en DataFrame y actualiza CSV
            push!(results_df, (filt_name, red_name, mod_name, acc_mean, f1_mean, b_acc_mean, b_acc_str), promote=true)
            CSV.write(output_file, results_df)
            
        catch e
            # Captura errores para no detener el flujo y registra fallo
            println("ERROR en $filt_name + $red_name + $mod_name: $e")
            push!(results_df, (filt_name, red_name, mod_name, NaN, NaN, NaN, "ERROR"))
            CSV.write(output_file, results_df)
        end
        
        # Libera memoria tras cada iteración
        GC.gc()
    end
    
    println("Experimento finalizado.")
    return results_df
end

run_models (generic function with 1 method)

# Definición de los diccionarios para los modelos básicos y selección de atributos

In [3]:
# Definición de diccionarios de configuración
dic_filtros = Dict(
    "Sin_Filtrado" => nothing,
    "ANOVA" => MyANOVAFilter(n_features=N_FEATURES),
    "Pearson" => MyPearsonFilter(n_features=N_FEATURES),
    "Spearman" => MySpearmanFilter(n_features=N_FEATURES),
    "Kendall" => MyKendallFilter(n_features=N_FEATURES),
    "MI" => MyMIFilter(n_features=N_FEATURES),
    "RFE" => MyRFEFilter(n_features=N_FEATURES)
)

dic_reducciones = Dict(
    "Sin reducción" => IdentityTransformer(),
    "PCA" => PCA(variance_ratio=0.95),
    "ICA" => ICA(outdim=2, maxiter=10000,tol=0.5),
    "LDA" => LDA(method=:whiten, outdim=5) 
)

dic_modelos = Dict(
    "NeuralNetwork_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,))),
    "NeuralNetwork_100" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,))),
    "NeuralNetwork_100_50" => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50))),
    
    "KNN_1" => KNNClassifier(K=1),
    "KNN_10" => KNNClassifier(K=10),
    "KNN_20" => KNNClassifier(K=20),

    "SVM_0.1" => ProbabilisticSVC(cost=0.1),
    "SVM_0.5" => ProbabilisticSVC(cost=0.5),
    "SVM_1.0" => ProbabilisticSVC(cost=1.0)
);

In [4]:
# Ejecutar y guardar
df_resultados_basicos = run_models(
    dic_filtros, 
    dic_reducciones, 
    dic_modelos, 
    "resultados_modelos_basicos.csv"
)

Iniciando ejecución: 9 experimentos totales.

Evaluando (1/9): [ANOVA] + [ICA] + [KNN_1]
Acc: 0.5917 | F1: 0.5906 | B_acc: 0.5901

Evaluando (2/9): [Kendall] + [LDA] + [KNN_10]
Acc: 0.9575 | F1: 0.9575 | B_acc: 0.9577

Evaluando (3/9): [MI] + [PCA] + [KNN_20]
Acc: 0.8941 | F1: 0.8933 | B_acc: 0.8931

Evaluando (4/9): [Pearson] + [Sin reducción] + [NeuralNetwork_100]
Acc: 0.8915 | F1: 0.8872 | B_acc: 0.8904

Evaluando (5/9): [RFE] + [ICA] + [NeuralNetwork_100_50]
Acc: 0.5899 | F1: 0.5534 | B_acc: 0.5877

Evaluando (6/9): [Sin_Filtrado] + [LDA] + [NeuralNetwork_50]
Acc: 0.8953 | F1: 0.8901 | B_acc: 0.898

Evaluando (7/9): [Spearman] + [PCA] + [SVM_0.1]
Acc: 0.9102 | F1: 0.9096 | B_acc: 0.9102

Evaluando (8/9): [ANOVA] + [Sin reducción] + [SVM_0.5]
Acc: 0.9339 | F1: 0.9336 | B_acc: 0.934

Evaluando (9/9): [Kendall] + [ICA] + [SVM_1.0]
Acc: 0.6715 | F1: 0.6625 | B_acc: 0.6702
Experimento finalizado.


Row,Filter,Reduction,Model,Accuracy_Mean,F1_Score,B_Accuracy_mean,B_accuracy_list
,String,String,String,Float64,Float64,Float64,String
1,ANOVA,ICA,KNN_1,0.591744,0.590583,0.590066,0.62677;0.55199;0.61759;0.57407;0.5792
2,Kendall,LDA,KNN_10,0.957523,0.957549,0.957705,0.96633;0.94923;0.95082;0.94778;0.97424
3,MI,PCA,KNN_20,0.894079,0.893302,0.893082,0.89989;0.87871;0.89793;0.91762;0.87073
4,Pearson,Sin reducción,NeuralNetwork_100,0.891472,0.887208,0.890358,0.93597;0.83727;0.91861;0.81833;0.94309
5,RFE,ICA,NeuralNetwork_100_50,0.589897,0.553397,0.587747,0.56748;0.56918;0.59915;0.55906;0.64939
6,Sin_Filtrado,LDA,NeuralNetwork_50,0.895274,0.890135,0.898031,0.9083;0.89819;0.922;0.92621;0.83326
7,Spearman,PCA,SVM_0.1,0.910158,0.909612,0.910245,0.92863;0.89265;0.92161;0.90977;0.89776
8,ANOVA,Sin reducción,SVM_0.5,0.933949,0.933566,0.933977,0.9595;0.91366;0.93447;0.92911;0.93167
9,Kendall,ICA,SVM_1.0,0.671483,0.662471,0.670239,0.67872;0.67142;0.69538;0.66259;0.64239


# Modelos de ensemble

In [3]:
# ==============================================================================
# CONFIGURACIÓN DE LOS EXPERIMENTOS DE ENSEMBLE
# ==============================================================================

# 1. Configuración de Filtros (Solo probaremos sin filtrar de momento)
dic_filtros_basico = Dict(
    "Sin_Filtrado" => nothing
)

# 2. Configuración de Reducciones (PCA 95% y Sin Reducción)
dic_reducciones_ensemble = Dict(
    "Sin_Reduccion" => IdentityTransformer(), # O 'nothing' si prefieres
    "PCA_95"        => PCA(variance_ratio=0.95)
)

# 3. Definición de Modelos Base para los Ensembles
# ----------------------------------------------------------------
# Base para Bagging: KNN
knn_base = KNNClassifier(K=5)

# Base para AdaBoost: SVM Lineal 
# Usamos SGDClassifier con loss="hinge" que ES un SVM lineal, 
# pero compatible con el AdaBoost de ScikitLearn.
svm_base = SKSGDClassifier(
    loss         = "hinge",   # Hinge loss = comportamiento de SVM
    penalty      = "l2",      # Regularización estándar
    alpha        = 0.0001,    
    random_state = SEED       # Para reproducibilidad
)

# 4. Diccionario de Modelos de Ensemble
dic_modelos_ensemble = Dict(
    # --- Bagging (KNN) ---
    "Bagging_KNN_10" => EnsembleModel(
        model = knn_base,
        n     = 10
    ),
    "Bagging_KNN_50" => EnsembleModel(
        model = knn_base,
        n     = 50
    ),

    # --- AdaBoost (SVM Lineal) ---
    "AdaBoost_SVM" => AdaBoostClassifier(
        estimator = svm_base,
        n_estimators   = 5,
        algorithm      = "SAMME" # Obligatorio para SVM/Hinge loss
    ),

    # --- EvoTrees (Gradient Boosting) ---
    "EvoTree_50" => EvoTreeClassifier(
        nrounds = 50, 
        eta     = 0.2
    ),
    "EvoTree_100" => EvoTreeClassifier(
        nrounds = 100, 
        eta     = 0.2
    )
)

Dict{String, Probabilistic} with 5 entries:
  "EvoTree_50"     => EvoTreeClassifier(loss = mlogloss, …)
  "AdaBoost_SVM"   => AdaBoostClassifier(estimator = SGDClassifier(loss = hinge…
  "Bagging_KNN_10" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "Bagging_KNN_50" => ProbabilisticEnsembleModel(model = KNNClassifier(K = 5, ……
  "EvoTree_100"    => EvoTreeClassifier(loss = mlogloss, …)

In [ ]:
run_experiment(
    dic_filtros_basico, 
    dic_reducciones_ensemble, 
    dic_modelos_ensemble, 
    "resultados_ensembles_bagging.csv" # Guardamos en un archivo nuevo
)

Archivo de checkpoint encontrado: resultados_ensembles_bagging.csv
1 experimentos completados previamente.

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_10]
Acc: 0.894 | F1: 0.8941 | B_acc: 0.894

Evaluando: [Sin_Filtrado] + [PCA_95] + [Bagging_KNN_50]
Acc: 0.8931 | F1: 0.8933 | B_acc: 0.8932

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_100]
Acc: 0.8893 | F1: 0.8882 | B_acc: 0.8888

Evaluando: [Sin_Filtrado] + [PCA_95] + [EvoTree_50]
Acc: 0.877 | F1: 0.8758 | B_acc: 0.8762

Evaluando: [Sin_Filtrado] + [Sin_Reduccion] + [AdaBoost_SVM]


┌ Error: Problem fitting the machine machine(:clf, …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695
┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704
┌ Error: Problem fitting machine(:clf, …)
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:767
┌ Error: Problem fitting the machine machine(PersonalizedPipeline(scaler = MyMinMaxScaler(), …), …). 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:695


ERROR en Sin_Filtrado + Sin_Reduccion + AdaBoost_SVM:
TaskFailedException

┌ Info: Running type checks... 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:700
┌ Info: Type checks okay. 
└ @ MLJBase C:\Users\Pc\.julia\packages\MLJBase\oYYNJ\src\machines.jl:704




    nested task error: Python: InvalidParameterError: The 'estimator' parameter of AdaBoostClassifier must be an object implementing 'fit' and 'predict' or None. Got Julia:
    SGDClassifier(
      loss = "hinge", 
      penalty = "l2", 
      alpha = 0.0001, 
      l1_ratio = 0.15, 
      fit_intercept = true, 
      max_iter = 1000, 
      tol = 0.001, 
      shuffle = true, 
      verbose = 0, 
      epsilon = 0.1, 
      n_jobs = nothing, 
      random_state = 104, 
      learning_rate = "optimal", 
      eta0 = 0.0, 
      power_t = 0.5, 
      early_stopping = false, 
      validation_fraction = 0.1, 
      n_iter_no_change = 5, 
      class_weight = nothing, 
      warm_start = false, 
      average = false) instead.
    Python stacktrace:
     [1] validate_parameter_constraints
       @ sklearn.utils._param_validation C:\Users\Pc\.julia\environments\v1.11\.CondaPkg\.pixi\envs\default\Lib\site-packages\sklearn\utils\_param_validation.py:95
     [2] _validate_params
       @ sk